# Giai đoạn 1 — EDA dữ liệu tạm (proxy dataset)

Mục tiêu: hiểu phân bố lớp, kiểm tra ảnh lỗi/trùng lặp trước khi build pipeline.
Xem kế hoạch chi tiết: `docs/proposal_summary.md`.

In [ ]:
# Nếu chạy trên Colab, bỏ comment 2 dòng dưới
# !git clone <repo-url> && %cd bitss-stool-classification
# !pip install -r requirements.txt -q

In [ ]:
import sys
sys.path.insert(0, 'src')

from dataset import class_distribution_report
import yaml

with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

root_dir = cfg['data']['root_dir']
class_names = cfg['data']['class_names']

## Phân bố số lượng ảnh theo lớp và theo split

Kiểm tra class imbalance — nếu lệch mạnh, cần class weighting hoặc oversampling ở Giai đoạn 3.

In [ ]:
df = class_distribution_report(root_dir, class_names)
df_pivot = df.pivot(index='class', columns='split', values='count')
df_pivot

In [ ]:
import matplotlib.pyplot as plt

df_pivot.plot(kind='bar', figsize=(10, 5))
plt.title('Phân bố số ảnh theo lớp và split')
plt.ylabel('Số ảnh')
plt.tight_layout()
plt.show()

## Kiểm tra ảnh lỗi / trùng lặp (hash-based)

Kiểm tra xem có ảnh nào bị trùng giữa train/valid/test không — nếu có, đây là
một dạng data leakage khác cần xử lý trước khi huấn luyện.

In [ ]:
import hashlib
from pathlib import Path
from collections import defaultdict

def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

hash_to_paths = defaultdict(list)
for split in ['train', 'valid', 'test']:
    for class_name in class_names:
        class_dir = Path(root_dir) / split / class_name
        if not class_dir.exists():
            continue
        for img_path in class_dir.glob('*.jpg'):
            hash_to_paths[file_hash(img_path)].append(str(img_path))

duplicates = {h: paths for h, paths in hash_to_paths.items() if len(paths) > 1}
print(f'Số nhóm ảnh trùng lặp phát hiện: {len(duplicates)}')
for h, paths in list(duplicates.items())[:5]:
    print(paths)

## Xem vài ảnh mẫu mỗi lớp

Giúp có cảm nhận trực quan về sự khác biệt giữa các mức BITSS trong dataset tạm.

In [ ]:
from PIL import Image

fig, axes = plt.subplots(1, len(class_names), figsize=(18, 3))
for ax, class_name in zip(axes, class_names):
    class_dir = Path(root_dir) / 'train' / class_name
    sample_imgs = list(class_dir.glob('*.jpg'))
    if sample_imgs:
        img = Image.open(sample_imgs[0])
        ax.imshow(img)
        ax.set_title(class_name)
    ax.axis('off')
plt.tight_layout()
plt.show()